In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
model = ChatOpenAI()

In [3]:
# Create Workflow which calculates batsman strike_rate, boundaries_per_balls, boundaries_run_percentage

class State(TypedDict):
    # input state
    runs: int
    balls: int
    fours: int
    sixes: int
    
    # workflow States
    sr: float
    bpb: float
    brp: float

In [ ]:
def calculate_sr(state: State) :
    sr = (state['runs']/state['balls']) * 100

    # return state -> don't return whole state in parallel nodes, because it assumes you are changing state all keys value. 
    # So it get confuse from which node need to take value
    return {"sr": sr}

def calculate_bpb(state: State) :
    bpb = state['balls']/(state['fours']+state['sixes'])
    return {"bpb": bpb}

def calculate_brp(state: State) :
    brp = (((state['fours']*4)+(state['sixes'] * 6))/state['runs'])*100
    return {"brp": brp}

def summery(state: State) :
    return {
        "summery": f'''
        Strike Rate: {state['sr']}
        Balls per Boundary: {state['bpb']}
        Boundary Run Percentage: {state['brp']}
        '''
    }

In [11]:
# define graph
graph = StateGraph(State)

# add Node
graph.add_node('calculate_sr', calculate_sr)
graph.add_node('calculate_bpb', calculate_bpb)
graph.add_node('calculate_brp', calculate_brp)
graph.add_node('summery', summery)

# add Edges

# Run edges parallel
graph.add_edge(START, "calculate_sr")
graph.add_edge(START, "calculate_bpb")
graph.add_edge(START, "calculate_brp")

# pass all above nodes result to summery
graph.add_edge("calculate_sr", "summery")
graph.add_edge("calculate_bpb", "summery")
graph.add_edge("calculate_brp", "summery")

graph.add_edge("summery", END)

# compile
workflow = graph.compile()

# invoke
initial_state = {
    "runs":100,
    "balls":50,
    "fours": 4,
    "sixes": 6
}

updated_state = workflow.invoke(initial_state)

In [12]:
print(updated_state)

{'runs': 100, 'balls': 50, 'fours': 4, 'sixes': 6, 'sr': 200.0, 'bpb': 5.0, 'brp': 52.0}
